In [2]:
from langchain.agents import create_agent
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

USER_DATABASE = {
  "school_a": {
    "user_a": {"name": "张三", "course": "langchain开发指南"},
    "user_b": {"name": "李四", "course": "java"},
  },
  "school_b": {
    "user_a": {"name": "王五", "course": "python基础"},
    "user_b": {"name": "丁六", "course": "fastapi构建"},
  }
}

@dataclass(frozen=True)
class UserContext:
  user_id: str
  tenant_id: str

@tool
def get_user_info(runtime: ToolRuntime[UserContext]):
  """获取当前登录用户的信息"""
  ctx = runtime.context
  print(ctx)
  profile = USER_DATABASE.get(ctx.tenant_id, {}).get(ctx.user_id)
  return f"姓名：{profile["name"]}，课程：{profile["course"]}"

agent = create_agent(
  model="deepseek-flash",
  tools=[get_user_info],
  context_schema=UserContext,
  system_prompt="你是课程平台助手，需要用户资料时调用工具"
)

response = agent.invoke(
  {"messages": [HumanMessage("你好，帮我查询一下我正在学习的课程")]},
  context=UserContext("user_a","school_b")
)

for message in response["messages"]:
  message.pretty_print()

UserContext(user_id='user_a', tenant_id='school_b')
================================ Human Message =================================

你好，帮我查询一下我正在学习的课程
================================== Ai Message ==================================

我先获取一下你的账户资料，稍等。
Tool Calls:
  get_user_info (call_00_RvcEEeAXiSUjxCkVRYrp8348)
 Call ID: call_00_RvcEEeAXiSUjxCkVRYrp8348
  Args:
================================= Tool Message =================================
Name: get_user_info

姓名：王五，课程：python基础
================================== Ai Message ==================================

你好，王五！查询到你的账号信息如下：

- **姓名**：王五
- **正在学习的课程**：**Python 基础**

你目前正在学习「Python 基础」这门课程。需要我帮你查看课程内容、学习进度，或者提供一些学习建议吗？
